In [2]:
import os
import pandas as pd

# Function to extract data from a filename
def extract_data_from_filename(filename, path):
    # Remove the extension
    filename = filename.replace('.qasm', '')
    
    # Split the filename into parts
    parts = filename.split('_')
    
    # Initialize a dictionary to hold the extracted data
    data = {
        'path': path,
        'algorithm': parts[0],
        'qubits': parts[1],
        'operator': parts[3],
        'gate': parts[4],
        'position': parts[5].replace('P', ''),
        'qubit': parts[6].replace('Q', ''),
        'params': ''
    }
        # Extract position and qubit
    if len(parts) > 7:
        data['params'] = parts[7].replace('.qasm', '')
    
    return data

# Directory containing the files
directory = r'C:\Users\Enaut\PycharmProjects\noiseAwareMutants\experiment\all_mutants'

# List to hold all extracted data
data_list = []

# Traverse the directory and process each file
for root, dirs, files in os.walk(directory):
    for file in files:
        if file.endswith('.qasm'):
            path = os.path.join(root, file)
            data = extract_data_from_filename(file, path)
            data_list.append(data)

# Create a DataFrame from the data list
df = pd.DataFrame(data_list, columns=['path','algorithm', 'qubits', 'operator', 'gate', 'position', 'qubit', 'params'])

# Display the DataFrame
df


,path,algorithm,qubits,operator,gate,position,qubit,params
0,C:\Users\Enaut\PycharmProjects\noiseAwareMutan...,ae,2,Add,cp,2,0,[0.7853981633974483]
1,C:\Users\Enaut\PycharmProjects\noiseAwareMutan...,ae,2,Add,cp,4,0,[5.497787143782138]
2,C:\Users\Enaut\PycharmProjects\noiseAwareMutan...,ae,2,Add,cx,2,0,[]
3,C:\Users\Enaut\PycharmProjects\noiseAwareMutan...,ae,2,Add,cx,4,0,[]
4,C:\Users\Enaut\PycharmProjects\noiseAwareMutan...,ae,2,Add,cz,2,0,[]
...,...,...,...,...,...,...,...,...
22954,C:\Users\Enaut\PycharmProjects\noiseAwareMutan...,wstate,9,Replace,z,4,4,[]
22955,C:\Users\Enaut\PycharmProjects\noiseAwareMutan...,wstate,9,Replace,z,5,5,[]
22956,C:\Users\Enaut\PycharmProjects\noiseAwareMutan...,wstate,9,Replace,z,6,6,[]
22957,C:\Users\Enaut\PycharmProjects\noiseAwareMutan...,wstate,9,Replace,z,7,7,[]


In [3]:

# Group by 'algorithm' and 'qubits', and then count occurrences
grouped_df = df.groupby(['algorithm', 'qubits'])
grouped_counts = grouped_df.size().reset_index(name='count')

# Display the grouped counts
grouped_counts


,algorithm,qubits,count
0,ae,2,172
1,ae,3,343
2,ae,5,757
3,ae,6,1000
4,ae,8,1558
5,ae,9,1873
6,qft,2,96
7,qft,3,168
8,qft,6,576
9,qft,8,960


In [ ]:
## SELECT THE REPRESENTATIVE SUBSET FOR EACH OPERATOR APART

# def minimal_subset_for_operator(group, operator):
#     # Filter the group for the specific operator
#     operator_group = group[group['operator'] == operator]
#     
#     # Determine all unique values for the columns of interest
#     unique_gate = set(group['gate'].unique())
#     unique_position = set(group['position'].unique())
#     unique_qubit = set(group['qubit'].unique())
#     
#     # Dictionaries to keep track of covered values
#     covered_gate = set()
#     covered_position = set()
#     covered_qubit = set()
#     
#     subset_indices = set()
#     
#     # Calculate coverage for the current operator
#     while not (covered_gate == unique_gate and covered_position == unique_position and covered_qubit == unique_qubit):
#         best_row = None
#         best_cover = 0
#         
#         # Evaluate the coverage of each row
#         for idx, row in operator_group.iterrows():
#             row_cover = 0
#             if row['gate'] not in covered_gate:
#                 row_cover += 1
#             if row['position'] not in covered_position:
#                 row_cover += 1
#             if row['qubit'] not in covered_qubit:
#                 row_cover += 1
#             
#             # Select the row that covers the most new values
#             if row_cover > best_cover:
#                 best_cover = row_cover
#                 best_row = idx
#         
#         if best_row is not None:
#             subset_indices.add(best_row)
#             # Update covered values
#             covered_gate.add(operator_group.loc[best_row, 'gate'])
#             covered_position.add(operator_group.loc[best_row, 'position'])
#             covered_qubit.add(operator_group.loc[best_row, 'qubit'])
#         else:
#             break
#     
#     # Return the DataFrame with the selected rows
#     return group.loc[list(subset_indices)]
# 
# def balanced_subset_within_group(group):
#     operator_types = group['operator'].unique()
#     subsets = {op: minimal_subset_for_operator(group, op) for op in operator_types}
#     
#     # Determine the smallest subset size among all operators
#     min_size = min(len(subset) for subset in subsets.values())
#     
#     # Collect subsets and balance them
#     balanced_df = pd.concat([subset.sample(min_size) for subset in subsets.values()])
#     
#     return balanced_df
# 
# # Apply the balanced subset function to each group
# balanced_dfs = grouped_df.apply(balanced_subset_within_group)
# 
# # Reset index to get the DataFrame with the selected rows
# balanced_dfs = balanced_dfs.reset_index(drop=True)
# 
# balanced_dfs

In [4]:
def cover_unique_values(group):
    # Unique values required for coverage
    unique_gate = set(group['gate'].unique())
    unique_position = set(group['position'].unique())
    unique_qubit = set(group['qubit'].unique())
    
    # Split group into subgroups based on operator
    subgroups = {op: group[group['operator'] == op] for op in group['operator'].unique()}
    
    # Dictionaries to keep track of covered values
    covered_gate = set()
    covered_position = set()
    covered_qubit = set()
    
    selected_indices = set()
    
    # Continue until all unique values are covered
    while not (covered_gate == unique_gate and covered_position == unique_position and covered_qubit == unique_qubit):
        # Select one row from each subgroup
        for op in subgroups:
            subgroup = subgroups[op]
            
            if not subgroup.empty:
                # Find the best row that covers the most new values
                best_row = None
                best_cover = 0
                
                for idx, row in subgroup.iterrows():
                    row_cover = 0
                    if row['gate'] not in covered_gate:
                        row_cover += 1
                    if row['position'] not in covered_position:
                        row_cover += 1
                    if row['qubit'] not in covered_qubit:
                        row_cover += 1
                    
                    if row_cover > best_cover:
                        best_cover = row_cover
                        best_row = idx
                
                if best_row is not None:
                    selected_indices.add(best_row)
                    # Update covered values
                    covered_gate.add(subgroup.loc[best_row, 'gate'])
                    covered_position.add(subgroup.loc[best_row, 'position'])
                    covered_qubit.add(subgroup.loc[best_row, 'qubit'])
                    
                    # Remove the selected row from the subgroup
                    subgroups[op] = subgroup.drop(best_row)
    
    # Return the DataFrame with the selected rows
    return group.loc[list(selected_indices)]

# Apply the coverage function to each group
balanced_dfs = grouped_df.apply(cover_unique_values)

# Reset index to get the DataFrame with the selected rows
balanced_dfs = balanced_dfs.reset_index(drop=True)

balanced_dfs

C:\Users\Enaut\AppData\Local\Temp\ipykernel_21404\3962308863.py:55: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  balanced_dfs = grouped_df.apply(cover_unique_values)


,path,algorithm,qubits,operator,gate,position,qubit,params
0,C:\Users\Enaut\PycharmProjects\noiseAwareMutan...,ae,2,Add,cp,2,0,[0.7853981633974483]
1,C:\Users\Enaut\PycharmProjects\noiseAwareMutan...,ae,2,Replace,ry,0,0,[0.0]
2,C:\Users\Enaut\PycharmProjects\noiseAwareMutan...,ae,2,Add,h,0,0,[]
3,C:\Users\Enaut\PycharmProjects\noiseAwareMutan...,ae,2,Replace,rz,0,0,[1.5707963267948966]
4,C:\Users\Enaut\PycharmProjects\noiseAwareMutan...,ae,2,Replace,sx,0,0,[]
...,...,...,...,...,...,...,...,...
1087,C:\Users\Enaut\PycharmProjects\noiseAwareMutan...,wstate,9,Add,h,2,2,[]
1088,C:\Users\Enaut\PycharmProjects\noiseAwareMutan...,wstate,9,Replace,s,20,2,[]
1089,C:\Users\Enaut\PycharmProjects\noiseAwareMutan...,wstate,9,Add,h,5,5,[]
1090,C:\Users\Enaut\PycharmProjects\noiseAwareMutan...,wstate,9,Add,z,19,3,[]


In [6]:

grouped = balanced_dfs.groupby(['algorithm', 'qubits'])

def count_unique_values(group):
    counts = {}
    for column in group.columns:
        if column not in ['algorithm', 'qubits']:
            counts[column] = group[column].value_counts().to_dict()
    return pd.Series(counts)

unique_counts = grouped.apply(count_unique_values).reset_index()

unique_counts


C:\Users\Enaut\AppData\Local\Temp\ipykernel_21404\1626830007.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  unique_counts = grouped.apply(count_unique_values).reset_index()


,algorithm,qubits,path,operator,gate,position,qubit,params
0,ae,2,{'C:\Users\Enaut\PycharmProjects\noiseAwareMut...,"{'Add': 9, 'Replace': 8, 'Remove': 4}","{'cp': 1, 's': 1, 'id': 1, 'cz': 1, 'u': 1, 'u...","{'0': 11, '2': 5, '6': 1, '1': 1, '3': 1, '4':...","{'0': 18, '1': 3}","{'[]': 10, '': 4, '[0.7853981633974483]': 2, '..."
1,ae,3,{'C:\Users\Enaut\PycharmProjects\noiseAwareMut...,"{'Add': 9, 'Replace': 8, 'Remove': 5}","{'u': 2, 'sx': 1, 'u3': 1, 'u2': 1, 'cx': 1, '...","{'0': 8, '11': 2, '9': 1, '13': 1, '10': 1, '5...","{'0': 13, '2': 5, '1': 4}","{'[]': 10, '': 5, '[0.7853981633974483]': 2, '..."
2,ae,5,{'C:\Users\Enaut\PycharmProjects\noiseAwareMut...,"{'Add': 11, 'Replace': 10, 'Remove': 10}","{'cp': 5, 'h': 4, 'cx': 3, 'u3': 2, 'x': 1, 'p...","{'28': 1, '23': 1, '9': 1, '14': 1, '18': 1, '...","{'4': 9, '0': 7, '1': 6, '2': 5, '3': 4}","{'[]': 13, '': 10, '[5.497787143782138]': 2, '..."
3,ae,6,{'C:\Users\Enaut\PycharmProjects\noiseAwareMut...,"{'Remove': 14, 'Add': 14, 'Replace': 13}","{'cp': 12, 'h': 7, 'u3': 3, 'cx': 2, 'y': 1, '...","{'19': 1, '39': 1, '7': 1, '26': 1, '27': 1, '...","{'5': 11, '0': 8, '1': 7, '2': 6, '3': 5, '4': 4}","{'[]': 16, '': 14, '[2.356194490192345]': 3, '..."
4,ae,8,{'C:\Users\Enaut\PycharmProjects\noiseAwareMut...,"{'Add': 22, 'Replace': 21, 'Remove': 21}","{'cp': 23, 'h': 17, 'cx': 5, 'u2': 2, 'x': 1, ...","{'22': 1, '59': 1, '12': 1, '5': 1, '13': 1, '...","{'7': 15, '0': 10, '1': 9, '2': 8, '3': 7, '4'...","{'[]': 26, '': 21, '[2.356194490192345]': 5, '..."
5,ae,9,{'C:\Users\Enaut\PycharmProjects\noiseAwareMut...,"{'Add': 26, 'Remove': 26, 'Replace': 25}","{'cp': 30, 'h': 21, 'cx': 7, 'u3': 2, 'u2': 1,...","{'52': 1, '49': 1, '5': 1, '65': 1, '62': 1, '...","{'8': 17, '0': 11, '1': 10, '2': 9, '3': 8, '4...","{'[]': 30, '': 26, '[0.7853981633974483]': 6, ..."
6,qft,2,{'C:\Users\Enaut\PycharmProjects\noiseAwareMut...,"{'Replace': 8, 'Add': 8, 'Remove': 2}","{'rxx': 1, 'id': 1, 'cz': 1, 'cx': 1, 'z': 1, ...","{'0': 11, '1': 4, '3': 2, '2': 1}","{'1': 15, '0': 3}","{'[]': 9, '[4.71238898038469]': 3, '[3.1415926..."
7,qft,3,{'C:\Users\Enaut\PycharmProjects\noiseAwareMut...,"{'Add': 8, 'Replace': 8, 'Remove': 3}","{'cp': 2, 'cz': 1, 'ry': 1, 'id': 1, 'cx': 1, ...","{'0': 10, '1': 3, '6': 2, '3': 1, '5': 1, '4':...","{'2': 14, '0': 3, '1': 2}","{'[]': 9, '': 3, '[2.356194490192345]': 2, '[0..."
8,qft,6,{'C:\Users\Enaut\PycharmProjects\noiseAwareMut...,"{'Add': 9, 'Replace': 9, 'Remove': 8}","{'cp': 8, 'cx': 2, 'p': 1, 'rxx': 1, 'x': 1, '...","{'0': 3, '20': 1, '6': 1, '9': 1, '18': 1, '22...","{'5': 8, '4': 5, '2': 4, '3': 4, '1': 3, '0': 2}","{'[]': 10, '': 8, '[3.141592653589793]': 3, '[..."
9,qft,8,{'C:\Users\Enaut\PycharmProjects\noiseAwareMut...,"{'Add': 14, 'Remove': 13, 'Replace': 13}","{'cp': 21, 'cx': 3, 'p': 1, 'sx': 1, 'h': 1, '...","{'16': 1, '18': 1, '11': 1, '17': 1, '23': 1, ...","{'7': 8, '6': 7, '5': 6, '4': 5, '3': 5, '2': ...","{'': 13, '[]': 11, '[5.497787143782138]': 3, '..."


In [5]:
import shutil
selected_mutants = balanced_dfs['path']
for path in selected_mutants:
        
    # Define the destination file path
    dest_file_path = path.replace('all_mutants','selected_mutant_qc')
    # Ensure the destination directory exists
    os.makedirs(os.path.dirname(dest_file_path), exist_ok=True)

    # Copy the file to the destination
    shutil.copy2(path, dest_file_path)
    